# Core 07 - Environment and Eval

Objetivo: ejecutar Tools y Agents reales dentro de `toolkit.environment`, resumir el episodio con `environment_summary` y evaluar el mismo agente contra un oracle independiente.

**Lugar en el modelo:** Environment añade tiempo mediante episodios y steps; Eval observa y califica un Executable sin convertirse en parte de su lógica.

**Evidencia exigida:** tres steps reales deben producir reward/lineage y el mismo conjunto de casos debe aprobar tanto un Agent como un System.

**Límite de la evidencia:** un eval demuestra sólo los casos, oracles y condiciones de reproducibilidad declarados; no prueba corrección universal.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| records | tres casos | Entradas con oracle independiente. |
| runtime | python-runtime | Ejecutar Tools y Agents reales de forma reproducible. |
| reward | salida contra oracle | Evaluar comportamiento, no la implementacion interna. |

## 1) Casos declarativos

Los registros son entradas y expectativas, no una implementacion paralela. `expected_public` es el oracle declarado por quien disena el eval; el agente no lo recibe.

In [ ]:
import agentic_systems as toolkit

records = [
    {"case_id": "public_tool", "symbol": "tool", "expected_public": True},
    {"case_id": "public_graph", "symbol": "graph", "expected_public": True},
    {"case_id": "unknown_symbol", "symbol": "not_a_public_symbol", "expected_public": False},
]

toolkit.show_json(records, title="Environment records")

## 2) Tool, runtime y agent

La tool consulta la superficie instalada. El runtime local hace el ejemplo reproducible sin simular un Provider externo.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)
agent = system.agent(
    name="environment_public_api_inspector",
    instructions="Ejecuta inspect_public_api para el simbolo recibido.",
    tools=[inspect_public_api],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=2, temperature=0.0),
)

## 3) Adaptar el agent al Environment

`build_agent_step_graph` es el puente publico: cada step ejecuta el agent y conserva su `RunResult` normalizado.

In [ ]:
def environment_input(state: dict) -> dict:
    return {
        "tool": "inspect_public_api",
        "input": {"symbol": state["row"]["symbol"]},
    }


def environment_output(result, _state: dict) -> dict:
    return {"agent_output": toolkit.agent_output(result, kind="environment_step")}


def reward_fn(state: dict, row: dict, _action, _environment) -> float:
    fields = state.get("agent_output", {}).get("fields", {})
    return 1.0 if fields.get("is_public") == row["expected_public"] else 0.0

step_graph = toolkit.build_agent_step_graph(
    agent,
    input=environment_input,
    output=environment_output,
    result_key="agent_result",
    trace="agent_trace",
    mode="eval",
)

## 4) Ejecutar un episodio

El loop usa la interfaz Gymnasium-shaped publica: `reset` y `step`. No ejecuta Tools por fuera del Agent.

In [ ]:
environment = toolkit.environment(
    records=records,
    name="public_api_environment",
    graph=step_graph,
    reward_fn=reward_fn,
    render_mode="history",
)

observation, info = environment.reset(seed=247)
while observation is not None:
    observation, reward, terminated, truncated, info = environment.step()
    if terminated or truncated:
        break

episode_summary = toolkit.environment_summary(environment)
assert len(environment.history) == len(records)
assert environment.current_step == len(records)
toolkit.show_json(episode_summary, title="Environment summary")

## 5) Lineage del episodio

El resumen y lineage se derivan del historial real; no hay `finalize_episode` local.

In [ ]:
environment_lineage = environment.lineage(
    question="Que simbolos pertenecen a la API publica instalada?",
    goal="Explicar un episodio ejecutado por Agentic Systems.",
    tags=["environment", "api"],
)
toolkit.show(environment_lineage, title="Environment Lineage")

## 6) Eval con oracle independiente

El expected procede de los registros declarados. La funcion evaluada solo ve `symbol`; nunca recibe `expected_public`.

`Evaluator.run` acepta cualquier objeto con `run(...)`. Aquí los mismos casos se aplican primero al Agent y después al System de una unidad; el oracle permanece idéntico.

In [ ]:
eval_cases = [
    {
        "name": row["case_id"],
        "input": {"tool": "inspect_public_api", "input": {"symbol": row["symbol"]}},
        "expected": {
            "must_call": ["inspect_public_api"],
            "data_contains": {
                "tool": "inspect_public_api",
                "ok": True,
                "symbol": row["symbol"],
                "is_public": row["expected_public"],
            },
        },
    }
    for row in records
]

toolkit.show_json(eval_cases, title="Eval cases")

In [ ]:
agent_report = toolkit.eval().run(
    agent,
    eval_cases,
    determinism="deterministic",
    seed=247,
    reproducibility_conditions=[
        "same declared oracle",
        "python-runtime",
        "same installed PUBLIC_API",
    ],
)

system_report = toolkit.eval().run(
    system,
    eval_cases,
    determinism="deterministic",
    seed=247,
    reproducibility_conditions=[
        "same declared oracle",
        "python-runtime",
        "same installed PUBLIC_API",
    ],
)

agent_summary = toolkit.eval_report_summary(agent_report)
system_summary = toolkit.eval_report_summary(system_report)
assert agent_report.ok and system_report.ok
assert agent_report.total == system_report.total == len(eval_cases)
toolkit.show_json(
    {"agent": agent_summary, "system": system_summary},
    title="Agent and System eval summaries",
)
toolkit.human_result(agent_report, title="Agent Eval report")
toolkit.human_result(system_report, title="System Eval report")
agent_report.raise_if_failed()
system_report.raise_if_failed()

## 7) API realmente ejercitada

La cobertura excluye callbacks de dominio y enumera solo la fachada publica.

In [ ]:
api_coverage = [
    "toolkit.tool",
    "toolkit.runtime",
    "toolkit.system",
    "system.agent",
    "toolkit.build_agent_step_graph",
    "toolkit.environment",
    "environment.reset",
    "environment.step",
    "toolkit.environment_summary",
    "environment.lineage",
    "toolkit.eval().run",
    "toolkit.eval_report_summary",
    "toolkit.human_result",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
    "toolkit.agent_output",
    "toolkit.show",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Environment and Eval API coverage")

## Resultado e interpretacion

Tres steps ejecutados, reward total 3.0 y dos evals 3/3: uno sobre el Agent y otro sobre el System. Si se cambia deliberadamente un expected, `raise_if_failed()` detiene el notebook.